# 🎬 Render 3D Blender EEVEE Next v2 (Colab Pro L4 / A100 Ready!) — *"Jangan Sampai Kunci Ini Jatuh!"*

### Pembaruan Utama v2:
1. **Desain Karakter 100% Dipertahankan Persis Preview 1** (Bima, Oren, Pedagang Rempah, Penumpang Trem, Kunci Emas & Kunci Perak tetap dari rig terbuka `characters/` — dirender tajam ke 3D alpha-cutout di dalam dunia Blender EEVEE Next!).
2. **Langit Cerah & Warna Vibrant** (bebas langit hitam; menggunakan 3-stop World Sky cerah + Mist Pass Atmospheric Haze di Compositor).
3. **Properti, Bangunan & Tanaman Berbasis Referensi Nyata + 15 Model 3D CC0 Asli Poly Haven + PBR 2K** (arsitektur Positano/Cinque Terre, tanaman Bougainvillea botanis, Trem Lisbon 28 *Remodelado*, Menara Jam Venesia *Torre dell'Orologio*, & Mercusuar Eropa).

### Cara Pakai:
1. Pilih **Runtime → Change runtime type → L4 GPU atau A100 GPU** (Colab Pro) — atau T4 GPU.
2. Isi `GITHUB_REPO_URL` di Sel 1.
3. Jalankan Sel 1 → Sel 2 → **Sel 3 (Render 12 Preview Screenshot dulu untuk dinilai!)**.
4. Jika preview sudah pas, jalankan **Sel 4** untuk render final 58 detik (1.740 frame) + audio narasi ElevenLabs.

In [ ]:
# === SEL 1: KONFIGURASI COLAB PRO (L4 / A100 / T4) & CEK GPU ===
GITHUB_REPO_URL = "https://github.com/USERNAME/REPO_ANDA.git"  # <-- Ganti dengan URL repo GitHub Anda!
REPO_BRANCH = "main"

USE_GOOGLE_DRIVE = True           # Simpan hasil .blend, frame & .mp4 otomatis ke Google Drive
DOWNLOAD_POLYHAVEN_ASSETS = True  # Unduh 15 model 3D CC0 nyata (tanaman, jam, meja, tong, lampu jalan) + PBR 2K!
PBR_RESOLUTION = "2k"             # "2k" untuk Colab Pro (L4/A100), atau "1k" lebih cepat

# MESIN RENDER: "CYCLES" direkomendasikan di Colab Pro L4/A100 (pakai RT-Core OPTIX -> ~2-4 detik/frame, bebas bug CPU llvmpipe!)
RENDER_ENGINE = "CYCLES"          # "CYCLES" (NVIDIA OptiX GPU Super Cepat!) atau "EEVEE"
FINAL_WIDTH = 1080
FINAL_HEIGHT = 1920
RENDER_SAMPLES = 24               # 24 samples + GPU OpenImageDenoise + Persistent Data BVH -> ~2.5-3.5 detik/frame di L4!

!nvidia-smi

import os
DRIVE_OUT_DIR = "/content/drive/MyDrive/KunciJatuh_Blender3D_v2"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
    print("Google Drive terhubung:", DRIVE_OUT_DIR)

In [ ]:
# === SEL 2: CLONE/PULL REPO GITHUB + INSTALL BLENDER 4.2 LTS + RESVG/PILLOW DI PYTHON BLENDER ===
import glob, os, re, subprocess, urllib.request

PROJECT_DIR = "/content/kunci_jatuh_3d"
if os.path.exists(PROJECT_DIR):
    print("Menarik update terbaru dari repo GitHub...")
    !git -C {PROJECT_DIR} pull --ff-only
else:
    assert "USERNAME/REPO_ANDA" not in GITHUB_REPO_URL, "Silakan isi GITHUB_REPO_URL di Sel 1 terlebih dahulu!"
    !git clone --depth 1 -b {REPO_BRANCH} {GITHUB_REPO_URL} {PROJECT_DIR}

!apt-get -qq update
!apt-get -qq install -y libxi6 libunwind8 libgl1-mesa-dri libegl1-mesa libopengl0 libxkbcommon0 libsm6 libxrender1 xvfb ffmpeg

BLENDER_DIR = "/content/blender_42"
BLENDER_BIN = os.path.join(BLENDER_DIR, "blender")
if not os.path.exists(BLENDER_BIN):
    base_url = "https://download.blender.org/release/Blender4.2/"
    tar_name = "blender-4.2.3-linux-x64.tar.xz"
    try:
        html = urllib.request.urlopen(base_url, timeout=15).read().decode("utf-8", errors="ignore")
        matches = re.findall(r"blender-4\.2\.\d+-linux-x64\.tar\.xz", html)
        if matches:
            tar_name = sorted(set(matches), key=lambda s: int(s.split(".")[2].split("-")[0]))[-1]
    except Exception as e:
        print("Gunakan versi fallback LTS:", tar_name, e)
    dl_url = base_url + tar_name
    print(f"Mengunduh {dl_url}...")
    !wget -q --show-progress {dl_url} -O /content/blender.tar.xz
    os.makedirs(BLENDER_DIR, exist_ok=True)
    !tar -xf /content/blender.tar.xz -C {BLENDER_DIR} --strip-components=1
    !rm -f /content/blender.tar.xz

# WAJIB: Install resvg-py & pillow ke dalam Python bawaan Blender 4.2 agar desain karakter asli Preview 1 terender tajam!
b_pythons = sorted(glob.glob(f"{BLENDER_DIR}/4.*/python/bin/python3*"))
assert b_pythons, "Python bawaan Blender tidak ditemukan!"
BLENDER_PY = b_pythons[0]
print("Memasang resvg-py & pillow ke Python Blender:", BLENDER_PY)
!{BLENDER_PY} -m pip install -q --upgrade resvg-py pillow numpy

!{BLENDER_BIN} --version

# Buat 10_nvidia.json untuk NVIDIA EGL (penting jika memakai EEVEE; untuk CYCLES pakai OPTIX langsung tanpa X11!)
os.makedirs("/usr/share/glvnd/egl_vendor.d", exist_ok=True)
nv_egl = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(nv_egl):
    with open(nv_egl, "w") as f:
        f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')
os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"] = nv_egl

if RENDER_ENGINE.upper() == "CYCLES":
    RUN_PREFIX = ""
    print("Mode Render: CYCLES GPU OPTIX (Direct NVIDIA L4/A100 RT-Core — nol overhead Xvfb/llvmpipe!)")
else:
    probe_py = (
        "import bpy; s=bpy.context.scene; "
        "e=[x.identifier for x in bpy.types.RenderSettings.bl_rna.properties['engine'].enum_items]; "
        "s.render.engine='BLENDER_EEVEE_NEXT' if 'BLENDER_EEVEE_NEXT' in e else 'BLENDER_EEVEE'; "
        "s.render.resolution_x=64; s.render.resolution_y=64; s.render.filepath='/tmp/eevee_probe.png'; "
        "bpy.ops.render.render(write_still=True); print('EEVEE_EGL_OK')"
    )
    res = subprocess.run([BLENDER_BIN, "-b", "--python-expr", probe_py], capture_output=True, text=True)
    RUN_PREFIX = "" if (res.returncode == 0 and "EEVEE_EGL_OK" in res.stdout) else "xvfb-run -a "
    print("Mode Render: EEVEE ->", "NVIDIA EGL Native" if not RUN_PREFIX else "Xvfb Fallback")

In [ ]:
# === SEL 3: BANGUN SCENE 3D REFERENSI NYATA + MODEL POLY HAVEN & RENDER 12 PREVIEW ===
import os, shutil, sys
from PIL import Image, ImageDraw
from IPython.display import display, Image as IPyImage

assets_flag = f"--download-assets --pbr-res {PBR_RESOLUTION}" if DOWNLOAD_POLYHAVEN_ASSETS else ""
cyc_flag = "--cycles-device OPTIX" if RENDER_ENGINE.upper() == "CYCLES" else ""
cmd = (
    f"{RUN_PREFIX}{BLENDER_BIN} -b {cyc_flag} -P {PROJECT_DIR}/blender_pipeline/render_eevee.py -- "
    f"--mode previews --engine {RENDER_ENGINE} --width 720 --height 1280 --samples {RENDER_SAMPLES} {assets_flag}"
)
print("Menjalankan:", cmd)
!{cmd}

sys.path.insert(0, PROJECT_DIR)
from blender_pipeline.config import HERO_PREVIEW_SHOTS, PREVIEW_DIR, OUTPUT_DIR

def build_colab_sheet(title, accent, shot_slice, out_name):
    cw, ch, gap, pad = 440, 782, 20, 26
    w = pad * 2 + cw * 4 + gap * 3
    h = 92 + ch + 78
    sheet = Image.new("RGB", (w, h), (18, 16, 26))
    draw = ImageDraw.Draw(sheet)
    draw.rectangle([0, 0, w, 76], fill=(14, 12, 22))
    draw.rectangle([0, 72, w, 76], fill=accent)
    draw.text((24, 24), title, fill=(248, 245, 255))
    for idx, (fname, _t, badge, desc) in enumerate(shot_slice):
        fp = os.path.join(PREVIEW_DIR, fname)
        if not os.path.exists(fp):
            continue
        x0, y0 = pad + idx * (cw + gap), 92
        im = Image.open(fp).convert("RGB").resize((cw, ch), Image.Resampling.LANCZOS)
        sheet.paste(im, (x0, y0))
        draw.rectangle([x0, y0, x0 + cw, y0 + ch], outline=accent, width=3)
        draw.rectangle([x0, y0 + ch + 6, x0 + cw, y0 + ch + 68], fill=(28, 24, 40))
        draw.text((x0 + 10, y0 + ch + 14), badge, fill=accent)
        draw.text((x0 + 10, y0 + ch + 36), desc[:58], fill=(230, 226, 240))
    out_p = os.path.join(PREVIEW_DIR, out_name)
    sheet.save(out_p, quality=94)
    if USE_GOOGLE_DRIVE:
        shutil.copy2(out_p, os.path.join(DRIVE_OUT_DIR, out_name))
    return out_p

sheets = [
    build_colab_sheet("EEVEE NEXT v2 SHEET 01 — FASE 1 LOW SUN CERAH (BENGKEL & PASAR)", (242, 183, 5), HERO_PREVIEW_SHOTS[0:4], "eevee_v2_sheet_01_fase1.png"),
    build_colab_sheet("EEVEE NEXT v2 SHEET 02 — GANG KAFE (OREN) & STORY TURN GORONG-GORONG", (46, 224, 166), HERO_PREVIEW_SHOTS[4:8], "eevee_v2_sheet_02_story_turn.png"),
    build_colab_sheet("EEVEE NEXT v2 SHEET 03 — KLIMAKS TREM LISBON & MENARA JAM VENESIA", (102, 240, 255), HERO_PREVIEW_SHOTS[8:12], "eevee_v2_sheet_03_climax_finale.png"),
]
if USE_GOOGLE_DRIVE:
    blend_src = os.path.join(OUTPUT_DIR, "jangan_sampai_kunci_ini_jatuh.blend")
    if os.path.exists(blend_src):
        shutil.copy2(blend_src, os.path.join(DRIVE_OUT_DIR, "jangan_sampai_kunci_ini_jatuh_v2.blend"))
        print("File .blend v2 tersalin ke Google Drive!")

for sp in sheets:
    display(IPyImage(filename=sp, width=1000))

In [ ]:
# === SEL 4: RENDER VIDEO FINAL 58 DETIK (1.740 FRAME) + MUXING AUDIO ELEVENLABS ===
# TIPS SUPER CEPAT (Colab Pro): Bisa buka 3-4 tab Colab paralel ke folder Google Drive yang SAMA!
#   Tab 1: START_FRAME = 1,    END_FRAME = 435
#   Tab 2: START_FRAME = 436,  END_FRAME = 870
#   Tab 3: START_FRAME = 871,  END_FRAME = 1305
#   Tab 4: START_FRAME = 1306, END_FRAME = 1740
# Frame yang sudah ada di Google Drive otomatis dilewati (--resume)! Tab terakhir otomatis mux MP4 + audio!
import os, shutil

START_FRAME = 1
END_FRAME = 1740

BLEND_FILE = f"{PROJECT_DIR}/output/jangan_sampai_kunci_ini_jatuh.blend"
if USE_GOOGLE_DRIVE and not os.path.exists(BLEND_FILE):
    for cand in ("jangan_sampai_kunci_ini_jatuh_v2.blend", "jangan_sampai_kunci_ini_jatuh.blend"):
        drv_blend = os.path.join(DRIVE_OUT_DIR, cand)
        if os.path.exists(drv_blend):
            os.makedirs(f"{PROJECT_DIR}/output", exist_ok=True)
            shutil.copy2(drv_blend, BLEND_FILE)
            print("Memuat .blend langsung dari Google Drive (tanpa build ulang!):", drv_blend)
            break
FINAL_MP4 = f"{PROJECT_DIR}/output/jangan_sampai_kunci_ini_jatuh_eevee_v2_{FINAL_WIDTH}x{FINAL_HEIGHT}.mp4"

if USE_GOOGLE_DRIVE:
    drive_frames = os.path.join(DRIVE_OUT_DIR, f"frames_{FINAL_WIDTH}x{FINAL_HEIGHT}")
    os.makedirs(drive_frames, exist_ok=True)
    local_frames = f"{PROJECT_DIR}/output/frames"
    if os.path.islink(local_frames):
        os.unlink(local_frames)
    elif os.path.isdir(local_frames):
        shutil.rmtree(local_frames)
    os.makedirs(f"{PROJECT_DIR}/output", exist_ok=True)
    os.symlink(drive_frames, local_frames)
    print("Folder frames diarahkan langsung ke Google Drive (auto-resume aman):", drive_frames)

assets_flag = f"--download-assets --pbr-res {PBR_RESOLUTION}" if DOWNLOAD_POLYHAVEN_ASSETS else ""
cyc_flag = "--cycles-device OPTIX" if RENDER_ENGINE.upper() == "CYCLES" else ""
cmd_vid = (
    f"{RUN_PREFIX}{BLENDER_BIN} -b {cyc_flag} -P {PROJECT_DIR}/blender_pipeline/render_eevee.py -- "
    f"--mode video --engine {RENDER_ENGINE} --blend-in {BLEND_FILE} "
    f"--width {FINAL_WIDTH} --height {FINAL_HEIGHT} --samples {RENDER_SAMPLES} "
    f"--start-frame {START_FRAME} --end-frame {END_FRAME} {assets_flag} "
    f"--resume --out-mp4 {FINAL_MP4}"
)
print("Menjalankan Render Final:", cmd_vid)
!{cmd_vid}

if os.path.exists(FINAL_MP4):
    if USE_GOOGLE_DRIVE:
        dst_mp4 = os.path.join(DRIVE_OUT_DIR, os.path.basename(FINAL_MP4))
        shutil.copy2(FINAL_MP4, dst_mp4)
        print("Video final tersimpan di Google Drive:", dst_mp4)
    from google.colab import files
    files.download(FINAL_MP4)